### Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic 

Pydantic models provide the richest feature set with field validation, descriptions and nested structures.

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000012B25514050>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000012B25514AD0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [8]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description = "The Title of the movie")
    year:int = Field(description = "The year movie was released")
    director:str = Field(description = "The director of the movie")
    rating:str = Field(description = "The movies' rating out of 10")
    plot:str = Field(description = "The plot of the movie")

In [9]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000012B25514050>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000012B25514AD0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': 

In [10]:
model.invoke("Provide details about movie inception")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know. It\'s a science fiction action film directed by Christopher Nolan, right? The main actor is Leonardo DiCaprio. The title "Inception" refers to the concept of planting an idea into someone\'s mind, which is a central theme of the movie.\n\nThe plot revolves around Dom Cobb, played by DiCaprio, who is a professional thief that steals secrets by infiltrating the subconscious of his targets. Instead of a theft, he is offered a chance to have his criminal history erased by performing the reverse: planting an idea into a target\'s mind. This is the inception part. The team has to navigate through multiple layers of dreams to accomplish this.\n\nThere\'s a lot of talk about dream-sharing technology. I think they use devices like masks or helmets to enter each other\'s dreams. The concept of limbo is also important, which is a place where the mind goes when the dream

In [11]:
model_with_structure.invoke("Provide details about movie inception")


Movie(title='Inception', year=2010, director='Christopher Nolan', rating='8.8', plot="A thief who enters the dreams of others to steal secrets is given a final task of planting an idea into a target's mind.")

### Message Alongside with Parsed Structure

In [12]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie wit details"""
    title:str = Field(..., description = "The Title of the movie")
    year:int = Field(..., description = "The year movie was released")
    director:str = Field(..., description = "The director of the movie")
    rating:str = Field(..., description = "The movies' rating out of 10")
    plot:str = Field(..., description = "The plot of the movie")

model_with_structure = model.with_structured_output(Movie, include_raw = True)

response = model_with_structure.invoke("Provide details about movie inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check if I have the right tool for that. The available tool is the Movie function, which requires title, year, director, rating, and plot. I need to make sure I can provide all those parameters for Inception.\n\nFirst, the title is obviously "Inception". The release year was 2010. The director is Christopher Nolan. The rating, I think it\'s around 8.8 on IMDb. The plot involves Dom Cobb entering people\'s dreams to steal secrets, and he\'s offered a chance to erase his criminal past by performing the reverse. I should structure all that into the function call. Let me double-check the details to ensure accuracy. Yeah, that\'s correct. So I\'ll format the JSON with those parameters.\n', 'tool_calls': [{'id': '0yda2k2cm', 'function': {'arguments': '{"director":"Christopher Nolan","plot":"Dom Cobb is a thief who enters the dreams of others to steal s

### Nested Structure

In [13]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name : str
    role : str

class MovieDetails(BaseModel):
    title : str
    year : int
    cast : list[Actor]
    genre : list[str]
    budget : float | None = Field(None, description = "Budet in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about movie inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames')], genre=['Action', 'Sci-Fi', 'Thriller'], budget=None)

### TypedDict

TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation

In [3]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie wit details."""
    title : Annotated[str, ..., "The title of the movie"]
    year : Annotated[int, ..., "The year movie was released"]
    director : Annotated[str, ..., "The director of the movie"]
    rating : Annotated[float, ..., "The ratings of the movie"]

model_with_typeddict = model.with_structured_output(MovieDict)
response = model_with_typeddict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [15]:
# Nested TypedDict

class Actor(TypedDict):
    name : str
    role : str

class MovieDetails(TypedDict):
    title : str
    year : int
    cast : list[Actor]
    genre : list[str]
    budget : float | None = Field(None, description = "Budet in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about movie avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genre': ['Action', 'Science Fiction', 'Adventure'],
 'title': 'Avengers',
 'year': 2012}

In [ ]:
# Information about what the model supports

model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

### Data Classes:

A data class is a class typically containing mainly data, although there aren't any restrictions. You create it using the @dataclass decorator

In [ ]:
# Pydantic

from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name : str = Field("The name of the person")
    email : str = Field("The email address of the person")
    phone : str = Field("The phone number of the person")

agent = create_agent(
    model = model,
    response_format = ContactInfo   # Auto Selects ProviderStartegy
)

result = agent.invoke({
    "messages" : [{"role": "user", "content": "Extract contact info from: Jon Doe, jon@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: Jon Doe, jon@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='dc11453d-0aa1-4e58-8e4d-7911c898b297'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given text: "Jon Doe, jon@example.com, (555) 123-4567". The tools provided include a ContactInfo function with parameters for name, email, and phone.\n\nFirst, I need to parse the input. The name is "Jon Doe", which should go into the name field. The email is clearly "jon@example.com". The phone number is formatted as "(555) 123-4567", which I can clean up by removing the parentheses and dashes if needed, but the tool\'s parameters don\'t specify formatting, so maybe just pass it as-is.\n\nI should check if all parts are correctly identified. Sometimes names can have middle names or other parts, but here it\'s straightforward. The email and phone number are

In [19]:
result['structured_response']

ContactInfo(name='Jon Doe', email='jon@example.com', phone='(555) 123-4567')

In [21]:
# TypedDict

from langchain.agents import create_agent

class ContactInfo(TypedDict):
    name : str = Field("The name of the person")
    email : str = Field("The email address of the person")
    phone : str = Field("The phone number of the person")

agent = create_agent(
    model = model,
    response_format = ContactInfo   # Auto Selects ProviderStartegy
)

result = agent.invoke({
    "messages" : [{"role": "user", "content": "Extract contact info from: Jon Doe, jon@example.com, (555) 123-4567"}]
})

result['structured_response']

{'name': 'Jon Doe', 'email': 'jon@example.com', 'phone': '(555) 123-4567'}

In [22]:
# DataClass

from dataclasses import dataclass

@dataclass
class ContactInfo:
    name : str # The name of the person
    email : str # The email address of the person
    phone : str # The phone number of the person

agent = create_agent(
    model = model,
    response_format = ContactInfo   # Auto Selects ProviderStartegy
)

result = agent.invoke({
    "messages" : [{"role": "user", "content": "Extract contact info from: Jon Doe, jon@example.com, (555) 123-4567"}]
})

result['structured_response']

ContactInfo(name='Jon Doe', email='jon@example.com', phone='(555) 123-4567')